# W11B · 9A · 프롬프트를 고쳐 실제 추천 요청하기
**딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2 · 2026-11-11**

예상 → 수계산 → 실행 → 한 곳 수정 → 이유 설명. `None`은 자신의 답이며 예제 변수와 분리돼 있습니다. Run all 성공과 문제 풀이 완료는 다릅니다. Colab에서 자신의 사본을 저장하세요.


In [ ]:
from __future__ import annotations
import os,sys,subprocess,json,hashlib
os.environ['CUDA_VISIBLE_DEVICES']='-1'
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
os.environ['TF_ENABLE_ONEDNN_OPTS']='0'
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w11b'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps,neural,llm] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.31.0'],check=True)
    for package,wanted in [('tensorflow','2.21.0'),('keras','3.15.1')]:
        if package in sys.modules and getattr(sys.modules[package],'__version__',None)!=wanted:
            raise RuntimeError('설치가 갱신되었습니다. 런타임을 재시작한 뒤 첫 셀부터 다시 실행하세요.')
from pathlib import Path
from typing import Callable
import numpy as np
import pandas as pd
from IPython.display import display
from luna_recsys.llm_runtime import prepare_runtime,CHAT_MODEL,EMBED_MODEL,MODEL_DIGESTS
from luna_recsys.llm_recommendation import (load_amazon_reviews,select_cohort,split_fm_observations,
    prepare_prompt_case,recommendation_schema,run_prompt)
from luna_recsys.llm_explainer import build_llm_explainer,export_displayed_table
CACHE=Path.home()/'.cache/luna-recsys/w11b'
print('Python',sys.version.split()[0],'수업 ref',COURSE_VERSION,'CPU / 실제 모델 / API 키 불필요')


### 준비 · 최초 다운로드를 시작하세요
공식 실행 파일과 두 모델은 첫 준비 때 약2.2GB를 내려받습니다. CPU·디스크·네트워크에 따라 수 분이 걸립니다. 같은 런타임에서는 검증한 캐시를 재사용합니다. Colab 기본 CPU를 사용하세요. 실패하면 오류를 확인하고 이 셀을 다시 실행하며, 가짜 응답이나 임의 벡터로 바꾸지 않습니다.

교재는 Ollama/Llama3.1을 사용합니다. 수업은 무료 CPU에서 실행하려고 Qwen2.5 0.5B와 Nomic v1.5(768차원)를 고정합니다. 모델과 표본이 다르므로 교재 수치 재현이라고 부르지 않습니다.


In [ ]:
client=prepare_runtime(CACHE/'runtime')
frame=select_cohort(load_amazon_reviews(CACHE/'data'))
print('공식 자료 / 교재 제공 자료와 다른 소표본:',json.dumps(frame.attrs,ensure_ascii=False))
print('실제 모델 digest:',MODEL_DIGESTS)
display(frame[['user_id','item_id','rating','timestamp']].head())


<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11b/course/notion/assets/w11b/03-cf-protocol.png" alt="예시 사용자와 대상 사용자의 과거·보류 분리" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11b/course/notion/assets/w11b/03-cf-protocol.png) · 교재 개념을 재구성한 독자 도해입니다.


In [ ]:
case_report=prepare_prompt_case(frame)
case,truth,seen=case_report['input'],case_report['truth'],case_report['seen']
print('대상',case['customer'],'후보',case['candidates'])
print('관측 행',len(case_report['observed_rows']),'보류 행',len(case_report['heldout_rows']))
print('후보 안 보류정답 비율',case_report['candidate_truth_coverage'])
assert set(case_report['observed_rows']).isdisjoint(case_report['heldout_rows'])
assert set(case_report['example_users']).isdisjoint(case_report['target_users'])


### 본문 실행 정의 · `build_prompt`
입력·반환값을 읽고 아래 본문에서 수정하세요. 수정 뒤 이 셀과 후속 계산·앱 조립 셀을 다시 실행합니다.


In [ ]:
# Initial source SHA-256: 869afd648d525a32247a3c1800b94e0252e0b8bcc953ba56c4bd39a054e3c3fa
def build_prompt(case: dict, method: str = 'CF', shots: int = 3, review_chars: int = 140) -> list[dict]:
    """Visible input construction; receives past-only input, never the target answers.

    CF uses example histories; CB uses training-product/past-preference reviews.
    Changing these fields changes the actual subsequent model request.
    """
    if method not in ('CF', 'CB') or not 0 <= shots <= 4 or not 1 <= review_chars <= 240:
        raise ValueError('Choose CF/CB, 0–4 examples and 1–240 review characters.')
    payload = {'customer': case['customer'], 'K': case['k'], 'method': method,
               'observed_history': case['history']}
    if method == 'CF':
        payload['examples'] = case['examples'][:shots]
        payload['catalog'] = case['candidates']
    else:
        payload['preferences'] = case['preference_reviews']
        payload['catalog'] = [{'product': iid, 'reviews': case['catalog_reviews'][iid][:review_chars]}
                              for iid in case['candidates']]
    instruction = ('Recommend K distinct product codes from the provided catalog. '
        'Use the other customers histories for CF, or the review contents and ratings for CB. '
        'Do not output customer codes. Return JSON with a recommendations list of product codes.')
    return [{'role': 'system', 'content': instruction},
            {'role': 'user', 'content': json.dumps(payload, ensure_ascii=False)}]


### 본문 실행 정의 · `validate_recommendations`
입력·반환값을 읽고 아래 본문에서 수정하세요. 수정 뒤 이 셀과 후속 계산·앱 조립 셀을 다시 실행합니다.


In [ ]:
# Initial source SHA-256: 750950af3d8b0ef698a8f41816907c52f05c3efa069de854e73eebee16a16aaf
def validate_recommendations(raw: str, candidates: list[str], seen: set[str], k: int = 3) -> dict:
    """Validate actual output, reporting errors instead of inventing replacement IDs."""
    if k < 1:
        raise ValueError('K must be positive.')
    try:
        obj = json.loads(raw)
        values = obj['recommendations']
        if not isinstance(values, list):
            raise ValueError('recommendations must be a list')
    except (ValueError, KeyError, TypeError) as exc:
        return {'status': 'invalid', 'valid': [], 'rejected': [{'reason': 'invalid-json-or-schema'}],
                'requested': k, 'returned': 0, 'detail': str(exc)}
    valid, rejected = [], []
    for value in values:
        reason = ('not-a-string' if not isinstance(value, str) else
                  'already-observed' if value in seen else
                  'outside-catalog' if value not in candidates else
                  'duplicate' if value in valid else
                  'beyond-requested-K' if len(valid) >= k else '')
        if reason:
            rejected.append({'value': value, 'reason': reason})
        else:
            valid.append(value)
    status = 'valid' if len(valid) == k and not rejected else 'partial' if valid else 'invalid'
    return {'status': status, 'valid': valid, 'rejected': rejected,
            'requested': k, 'returned': len(values)}


### 본문 실행 정의 · `ranking_overlap`
입력·반환값을 읽고 아래 본문에서 수정하세요. 수정 뒤 이 셀과 후속 계산·앱 조립 셀을 다시 실행합니다.


In [ ]:
# Initial source SHA-256: a54be56afe947fddb4e5a9bf55a2bf89ebe3f4e170db113f9213d3211ae28518
def ranking_overlap(recommendations: list[str], truth: list[str], k: int = 3) -> dict:
    """Observed-future overlap, not calibrated probability or proven satisfaction.

    Precision's denominator stays requested K even when output is short/invalid.
    """
    if k < 1 or not truth:
        raise ValueError('Positive K and a nonempty held-out set are required.')
    hits = len(set(recommendations[:k]) & set(truth))
    precision, recall = hits / k, hits / len(set(truth))
    return {'hits': hits, 'precision': precision, 'recall': recall,
            'f1': 2 * precision * recall / (precision + recall) if precision + recall else 0.,
            'requested_k': k, 'valid_count': len(set(recommendations[:k]))}


### P01 · 모델을 부르기 전에 입력을 읽기
CF 예시를3명에서0명으로 바꾸면 어떤 필드만 달라질까요? CB는 어떤 텍스트를 읽나요? 대상 마지막3개의 정답이 요청에 섞이면 왜 부정한 비교일까요? 사용자ID와 제품ID의 역할도 설명하세요.


In [ ]:
cf_messages=build_prompt(case,method='CF',shots=3)
cb_messages=build_prompt(case,method='CB',review_chars=140)
print('CF 입력 문자:',sum(len(m['content']) for m in cf_messages))
print('CB 입력 문자:',sum(len(m['content']) for m in cb_messages))
display(pd.DataFrame([{'role':m['role'],'내용 앞부분':m['content'][:400]} for m in cf_messages]))
my_P01=None


### 실제 CF·CB 요청 · 가중치 갱신은 없음
아래는 로컬 서버의 실제 생성 요청입니다. JSON 형식 준수와 정답 적중을 따로 읽습니다. 작은 모델·8개 후보·한 대상 결과로 방법의 우열을 결론 내리지 않습니다.


In [ ]:
results={}
for method in ['CF','CB']:
    result=run_prompt(case,truth,seen,client,method,prompt_fn=build_prompt,
                      validator_fn=validate_recommendations,metric_fn=ranking_overlap)
    results[method]=result
    print(method,'원래 응답:',result['raw'])
    print('검사',result['validation'],'보류제품 지표',result['metrics'])
    print('실제 토큰/시간',result['metadata'])
    display(result['table'])
comparison=pd.DataFrame([dict(method=m,**r['metrics'],prompt_chars=r['prompt_chars']) for m,r in results.items()])
display(comparison)


<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11b/course/notion/assets/w11b/06-response-validation.png" alt="JSON·후보·중복·관측 여부와 적중을 따로 검사" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11b/course/notion/assets/w11b/06-response-validation.png) · 교재 개념을 재구성한 독자 도해입니다.


### P02 · 잘못된 응답을 손으로 거르기
후보 A/B/C, 이미 관측한 S, K=3일 때 응답 `[A,A,U1,S,B]`를 검사합니다. 유효 목록은 무엇이며, 정답이 A/C이면 P@3과 recall은 얼마인가요? 실제 요청에는 가명 P코드를 쓰며 아래는 별도의 합성 검산입니다.


In [ ]:
diagnostic=validate_recommendations(json.dumps({'recommendations':['A','A','U1','S','B']}),['A','B','C'],{'S'},3)
print(diagnostic)
print(ranking_overlap(diagnostic['valid'],['A','C'],3))
my_P02=None


### P03 · 본문 코드를 실제로 수정하기
위 `build_prompt`의 CF 예시를 역순으로 선택하도록 한 줄 바꿔 다시 정의합니다. 같은 CF·shots3에서 `run_prompt`를 다시 실행하고 요청 해시·유효 목록·적중·입력 길이를 기록하세요. 결과가 같아도 입력이 달라졌는지 확인합니다. CB의 review_chars140→60은 별도 실험으로 진행하세요. 한 번에 한 조건만 바꿉니다.


In [ ]:
my_P03=None


### 누적 앱 · 본문 함수가 실제 요청과 CSV로 이어집니다
아래 앱에서 실제 요청 버튼을 누릅니다. 공유 주소는 런타임 종료 시 만료됩니다. 원문 리뷰는 앱에 표시하지 않습니다. CSV는 마지막 표시 표를 저장하므로 저장을 눌러도 새 언어모델 요청이 발생하지 않습니다.


In [ ]:
csv_path=export_displayed_table(results['CF']['table'])
pd.testing.assert_frame_equal(pd.read_csv(csv_path),results['CF']['table'],check_dtype=False)
demo=build_llm_explainer(client=client,case=case,truth=truth,seen=seen,
    prompt_fn=build_prompt,validator_fn=validate_recommendations,metric_fn=ranking_overlap)
if IN_COLAB:
    demo.queue(default_concurrency_limit=1).launch(share=True,inline=False,prevent_thread_lock=True)
print('W11B_PROMPT_RUNTIME_PASS',len(client.calls),'실제 요청 / 임의 응답 대체 없음')


출구 기록: ①입력에 들어간 근거 ②고정된 가중치 ③형식 실패와 적중 실패의 차이 ④한 번 수정한 코드와 관측 결과를 한 문장씩 쓰세요.


In [ ]:
assert all(result['raw'].strip() and not result['table'].empty for result in results.values())
assert all(set(result['table']['product'])<=set(case['candidates']) for result in results.values())
assert all(set(result['table']['product']).isdisjoint(seen) for result in results.values())
assert any(call.get('endpoint')=='chat' for call in client.calls)
print('W11B_PROMPT_CHECKS_PASS')
